In [3]:
import pandas as pd
import numpy as np
import torch
import json
from pathlib import Path

In [50]:
# ------------------------------------------------------------
# SMILES-Zeichen-Wörterbuch wie bei AttentionDTA
# ------------------------------------------------------------

CHARISOSMISET = {
    "#": 29, "%": 30, ")": 31, "(": 1, "+": 32, "-": 33, "/": 34, ".": 2,
    "1": 35, "0": 3, "3": 36, "2": 4, "5": 37, "4": 5, "7": 38, "6": 6,
    "9": 39, "8": 7, "=": 40, "A": 41, "@": 8, "C": 42, "B": 9, "E": 43,
    "D": 10, "G": 44, "F": 11, "I": 45, "H": 12, "K": 46, "M": 47, "L": 13,
    "O": 48, "N": 14, "P": 15, "S": 49, "R": 16, "U": 50, "T": 17, "W": 51,
    "V": 18, "Y": 52, "[": 53, "Z": 19, "]": 54, "\\": 20, "a": 55, "c": 56,
    "b": 21, "e": 57, "d": 22, "g": 58, "f": 23, "i": 59, "h": 24, "m": 60,
    "l": 25, "o": 61, "n": 26, "s": 62, "r": 27, "u": 63, "t": 28, "y": 64
}


# ------------------------------------------------------------
# SMILES-Encoding bleibt unverändert
# ------------------------------------------------------------

def label_smiles(smiles, max_len=100):
    """
    Encodiert einen SMILES-String zeichenweise mit dem gleichen
    CHARISOSMISET-Wörterbuch wie im AttentionDTA-Baseline-Modell.
    """
    encoded = np.zeros(max_len, dtype=np.int64)

    for i, char in enumerate(str(smiles)[:max_len]):
        encoded[i] = CHARISOSMISET.get(char, 0)

    return encoded

In [51]:


#load data from CSV with unicode encoding
train_df = pd.read_csv("C:\\Users\\hempe\\Studium\\Masterthesis\\Repository\\Masterthesis\\data\\processed\\train_data.csv", sep=',')

In [52]:
train_df.head()

,Ligand SMILES,BindingDB Target Chain Sequence 1,IC50 (nM)
0,CC(C)Nc1cccnc1N1CCN(CC1)C(=O)c1cc2ccc(C=O)cc2[...,PISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGK...,1400.0
1,[O-][N+](=O)c1ccc2N(Cc3ccccc3)C(=O)C(=O)c2c1,MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHL...,99500.0
2,CCOC(=O)c1c(Cc2cccc(Cl)c2)[nH]c2c1cc(O)c1ncccc21,MPSYTVTVATGSQWFAGTDDYIYLSLVGSAGCSEKHLLDKPFYNDF...,580.0
3,ON1C(=O)C(=C(C1=O)c1c[nH]c2ccccc12)c1c[nH]c2cc...,MADPAAGPPPSEGEESTVRFARKGALRQKNVHEVKNHKFTARFFKQ...,28900.0
4,c1c([nH]c2nccnc12)-c1cccc2ccccc12,MSGRPRTTSFAESCKPVQQPSAFGSMKVSRDKDGSKVTTVVATPGQ...,27000.0


In [53]:
sequence = train_df["Ligand SMILES"].iloc[2]

In [54]:
sequence_long = 'CC(C)Nc1cccnc1N1CCN(CC1)C(=O)c1cc2ccc(C=O)cc2[nH]1[O-][N+](=O)c1ccc2N(Cc3ccccc3)C(=O)C(=O)c2c1CCOC(=O)c1c(Cc2cccc(Cl)c2)[nH]c2c1cc(O)c1ncccc21'

In [55]:
sequence

'CCOC(=O)c1c(Cc2cccc(Cl)c2)[nH]c2c1cc(O)c1ncccc21'

In [56]:
long_eight_mers_list = []

for i in range(0, len(sequence_long)-7, 1):
    eight_mer = sequence_long[i:i+8]
    long_eight_mers_list.append(eight_mer)  

In [57]:
long_eight_mers_list

['CC(C)Nc1',
 'C(C)Nc1c',
 '(C)Nc1cc',
 'C)Nc1ccc',
 ')Nc1cccn',
 'Nc1cccnc',
 'c1cccnc1',
 '1cccnc1N',
 'cccnc1N1',
 'ccnc1N1C',
 'cnc1N1CC',
 'nc1N1CCN',
 'c1N1CCN(',
 '1N1CCN(C',
 'N1CCN(CC',
 '1CCN(CC1',
 'CCN(CC1)',
 'CN(CC1)C',
 'N(CC1)C(',
 '(CC1)C(=',
 'CC1)C(=O',
 'C1)C(=O)',
 '1)C(=O)c',
 ')C(=O)c1',
 'C(=O)c1c',
 '(=O)c1cc',
 '=O)c1cc2',
 'O)c1cc2c',
 ')c1cc2cc',
 'c1cc2ccc',
 '1cc2ccc(',
 'cc2ccc(C',
 'c2ccc(C=',
 '2ccc(C=O',
 'ccc(C=O)',
 'cc(C=O)c',
 'c(C=O)cc',
 '(C=O)cc2',
 'C=O)cc2[',
 '=O)cc2[n',
 'O)cc2[nH',
 ')cc2[nH]',
 'cc2[nH]1',
 'c2[nH]1[',
 '2[nH]1[O',
 '[nH]1[O-',
 'nH]1[O-]',
 'H]1[O-][',
 ']1[O-][N',
 '1[O-][N+',
 '[O-][N+]',
 'O-][N+](',
 '-][N+](=',
 '][N+](=O',
 '[N+](=O)',
 'N+](=O)c',
 '+](=O)c1',
 '](=O)c1c',
 '(=O)c1cc',
 '=O)c1ccc',
 'O)c1ccc2',
 ')c1ccc2N',
 'c1ccc2N(',
 '1ccc2N(C',
 'ccc2N(Cc',
 'cc2N(Cc3',
 'c2N(Cc3c',
 '2N(Cc3cc',
 'N(Cc3ccc',
 '(Cc3cccc',
 'Cc3ccccc',
 'c3ccccc3',
 '3ccccc3)',
 'ccccc3)C',
 'cccc3)C(',
 'ccc3)C(=',
 'cc3)C(=O',

In [4]:
eight_mers_list = []

for i in range(0, len(sequence)-7, 1):
    eight_mer = sequence[i:i+8]
    eight_mers_list.append(eight_mer)   

NameError: name 'sequence' is not defined

In [59]:
eight_mers_list

['CCOC(=O)',
 'COC(=O)c',
 'OC(=O)c1',
 'C(=O)c1c',
 '(=O)c1c(',
 '=O)c1c(C',
 'O)c1c(Cc',
 ')c1c(Cc2',
 'c1c(Cc2c',
 '1c(Cc2cc',
 'c(Cc2ccc',
 '(Cc2cccc',
 'Cc2cccc(',
 'c2cccc(C',
 '2cccc(Cl',
 'cccc(Cl)',
 'ccc(Cl)c',
 'cc(Cl)c2',
 'c(Cl)c2)',
 '(Cl)c2)[',
 'Cl)c2)[n',
 'l)c2)[nH',
 ')c2)[nH]',
 'c2)[nH]c',
 '2)[nH]c2',
 ')[nH]c2c',
 '[nH]c2c1',
 'nH]c2c1c',
 'H]c2c1cc',
 ']c2c1cc(',
 'c2c1cc(O',
 '2c1cc(O)',
 'c1cc(O)c',
 '1cc(O)c1',
 'cc(O)c1n',
 'c(O)c1nc',
 '(O)c1ncc',
 'O)c1nccc',
 ')c1ncccc',
 'c1ncccc2',
 '1ncccc21']

In [60]:
    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

In [61]:
for kmer in long_eight_mers_list:
    if kmer not in vocab:
        vocab[kmer] = len(vocab)

In [62]:
len(vocab)

135

In [63]:
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 'CC(C)Nc1': 2,
 'C(C)Nc1c': 3,
 '(C)Nc1cc': 4,
 'C)Nc1ccc': 5,
 ')Nc1cccn': 6,
 'Nc1cccnc': 7,
 'c1cccnc1': 8,
 '1cccnc1N': 9,
 'cccnc1N1': 10,
 'ccnc1N1C': 11,
 'cnc1N1CC': 12,
 'nc1N1CCN': 13,
 'c1N1CCN(': 14,
 '1N1CCN(C': 15,
 'N1CCN(CC': 16,
 '1CCN(CC1': 17,
 'CCN(CC1)': 18,
 'CN(CC1)C': 19,
 'N(CC1)C(': 20,
 '(CC1)C(=': 21,
 'CC1)C(=O': 22,
 'C1)C(=O)': 23,
 '1)C(=O)c': 24,
 ')C(=O)c1': 25,
 'C(=O)c1c': 26,
 '(=O)c1cc': 27,
 '=O)c1cc2': 28,
 'O)c1cc2c': 29,
 ')c1cc2cc': 30,
 'c1cc2ccc': 31,
 '1cc2ccc(': 32,
 'cc2ccc(C': 33,
 'c2ccc(C=': 34,
 '2ccc(C=O': 35,
 'ccc(C=O)': 36,
 'cc(C=O)c': 37,
 'c(C=O)cc': 38,
 '(C=O)cc2': 39,
 'C=O)cc2[': 40,
 '=O)cc2[n': 41,
 'O)cc2[nH': 42,
 ')cc2[nH]': 43,
 'cc2[nH]1': 44,
 'c2[nH]1[': 45,
 '2[nH]1[O': 46,
 '[nH]1[O-': 47,
 'nH]1[O-]': 48,
 'H]1[O-][': 49,
 ']1[O-][N': 50,
 '1[O-][N+': 51,
 '[O-][N+]': 52,
 'O-][N+](': 53,
 '-][N+](=': 54,
 '][N+](=O': 55,
 '[N+](=O)': 56,
 'N+](=O)c': 57,
 '+](=O)c1': 58,
 '](=O)c1c': 

In [64]:
eight_mers_encoded_list = []

for kmer in eight_mers_list:
    if kmer in vocab:
        eight_mers_encoded_list.append(vocab[kmer])
    else:
        eight_mers_encoded_list.append(vocab["<UNK>"])


In [65]:
for i in eight_mers_encoded_list:
    if i != 1:
        print(i)


95
96
97
26
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134


In [71]:
eight_mers_list[:10]

['CCOC(=O)',
 'COC(=O)c',
 'OC(=O)c1',
 'C(=O)c1c',
 '(=O)c1c(',
 '=O)c1c(C',
 'O)c1c(Cc',
 ')c1c(Cc2',
 'c1c(Cc2c',
 '1c(Cc2cc']

In [34]:
# Load C:\Users\hempe\Studium\Masterthesis\Repository\Masterthesis\data\encoded\encoded_train_smiles_8mer.pt


encoded_train_path = r"C:\Users\hempe\Studium\Masterthesis\Repository\Masterthesis\data\encoded\encoded_train_smiles_8mer.pt"
encoded_train = torch.load(encoded_train_path)

# Show first example of the encoded train data
print(type(encoded_train))
print(encoded_train.keys())


<class 'dict'>
dict_keys(['X_smiles', 'X_protein', 'y'])


In [35]:
encoded_train["X_smiles"].shape

torch.Size([43939, 100])

In [ ]:
encoded_train["X_protein"].shape

torch.Size([43939, 1200])

: 

In [28]:
print(encoded_train['X_smiles'][2])

tensor([81, 82, 83,  ...,  0,  0,  0])


In [20]:
idx = 0

print("SMILES encoded:")
print(data["X_smiles"][idx])

print("Protein encoded:")
print(data["X_protein"][idx])

print("Label / pIC50:")
print(data["y"][idx])

SMILES encoded:


NameError: name 'data' is not defined